<a href="https://colab.research.google.com/github/nmhom/MAT-422/blob/main/mat422_HW_1.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

HW 1.2

In [2]:
import numpy as np
from scipy.linalg import null_space

## 1.2.1 Linear Subspace
This section demonstrates the core definitions behind linear subspaces: linear independence, span, and closure under addition/scalar multiplication.



### Linear Independence

**Definition 1.2.5 & Lemma 1.2.6.** A set of vectors {u1,...,um} is linearly independent if none of them can be written as a combination of the others. Equivalently (Lemma 1.2.6), if A has the vectors as columns, the vectors are independent iff Ax = 0 has only the trivial solution x = 0, which holds exactly when rank(A) equals the number of columns.

Below, u = [1,0] and v = [0,1] are independent since neither can be built from the other (they point in different directions)

In [3]:
# Declare vectors
u = np.array([1,0])
v = np.array([0,1])

A = np.column_stack([u,v])

# Check the rank to determine if these vectors are independent
rank = np.linalg.matrix_rank(A)
is_independent = (rank == A.shape[1])

print("These vectors are independent:", is_independent)

These vectors are independent: True


### Span

**Definition 1.2.2.** The span of w1,..,wm is the set of all linearl combinations Σ(αj·wj). A vector w is in span(A) exactly when Ax = w has a solution. This is checked here by solving the system with least squares and verifying the reconstructed vector matches w.

Since u and v span all of R^2, w = [3,5] should be reachable.

In [4]:
# vector we want to check:
w = np.array([3,5])
# Checking if a vector is in span:
x, residuals, rank, s = np.linalg.lstsq(A, w, rcond=None)

# Then, multiple matrix A by vector x
result = A @ x

# Check result so it's a clear yes or no
print("Is w in span(A)?", np.allclose(result, w))

Is w in span(A)? True


### Subspace Closure

**Definition 1.2.1.** A linear subspace U is closed under addition and scalar multiplication: for u1, u2 ∈ U and α ∈ R, both u1+u2 ∈ U and αu1 ∈ U.

This is tested on the nullspace of B = [[1,1],[1,1]]. Two basis vectors of the nullspace are combined with a scalar, and we verify the result still satisfies Bx = 0, confirming it has not left the subspace.


In [5]:
ub = np.array([1,1])
vb = np.array([1,1])

B = np.column_stack([ub,vb])
basis = null_space(B)

# Pick a random scalar, alph
alph = 2

# Check if the null space has no basis vectors
if basis.shape[1] == 0:
    # if it has no basis vectors, it is closed under addition and scalar multiplication.
    is_closed = True
else:
    # Else, pick basis vectors
    u1 = basis[:, 0]
    u2 = basis[:, 0] if basis.shape[1] == 1 else basis[:, 1]

    # Compute alph * u1 + u2
    combo = alph * u1 + u2
    result = B @ combo

    # does B @ (alph * u1 + u2) still come out to approx. 0?
    is_closed = np.allclose(result, np.zeros_like(result))

print("The subspace is closed under addition and scalar multiplication: ", is_closed)

The subspace is closed under addition and scalar multiplication:  True


## 1.2.2 Orthogonality

Two vectors are orthogonal if they're perpendicular to each other (their dot product is zero). This section demonstrates the core definitions behind orthogonality: checking if two vectors are orthogonal, checking if a vector has norm 1, checking if a whole list of vectors is orthonormal, and computing an orthogonal projection.

### Checking if two vectors are orthogonal

**Definition 1.2.11.** A list of vectors is orthonormal if they are pairwise orthogonal (<u1, uj>) = 0 for i ≠ j) and each has norm 1. Here we check that two vectors are orthogonal exactly when their dot product is 0.

In [6]:
dot_prod = np.dot(u,v)
is_orthogonal = np.isclose(dot_prod, 0)
print("These vectors are orthogonal: ", is_orthogonal)

These vectors are orthogonal:  True


### Checking if a vector has norm 1

**Definition 1.2.10.** The norm of a vector is ||u|| = sqrt(Σ ui²). A vector is a unit vector if its norm equals 1, checked directly with np.linalg.norm.

In [7]:
norm = np.linalg.norm(u)
is_unit = np.isclose(norm, 1)
print("This vector has norm 1: ", is_unit)

This vector has norm 1:  True


### Checking if a whole list of vectors is orthonormal

**Definition 1.2.11**, combined with the identity **Q^TQ = I**. If Q's columns are orthonormal, then Q^TQ collapses to the identity matrix: diagonal entries are <qi, qi> = 1 (unit norm), and off-diagonal entries are <qi, qj> = 0 (orthogonality), so checking Q^TQ = I verifies both conditions for every vector in the list at once.

In [8]:
# If A's columns are truly orthonormal, then A^T @ A should equal the identity matrix
AtA = A.T @ A

# Answers if every vector have length 1 and if all the vectors are perpendicular to each other
is_orthonormal = np.allclose(AtA, np.eye(A.shape[1]))
print("Is A orthonormal: ", is_orthonormal)

Is A orthonormal:  True


### Computing an orthogonal projection

**Definition 1.2.17.** The orthogonal projection of v onto a subspace with orthonormal basis q1,...,qm is P(v) = Σ⟨v,qj⟩qj. For a single non-unit vector a (not necessarily normalized), the equivalent formula is ((v·a)/(a·a))·a.

In [9]:
# Using two non-orthogonal vectors
ortho_u = np.array([3,4])
ortho_v = np.array([2,0])
projection = (np.dot(ortho_v, ortho_u) / np.dot(ortho_u,ortho_u)) * ortho_u
print("Projection: ", projection)

Projection:  [0.72 0.96]


## 1.2.3 Gram-Schmidt process

The Gram-Schmidt process is a method for taking a set of linearly independent vectors and turning them into orthogonal (or orthonormal) vectors that span the same space.

### Orthogonalizing two vectors

**Theorem 1.2.20** Given linearly independent vactors a1,...,am, there exists an orthonormal basis q1,...,qm of span(a1,...,am). The process builds this basis one vector at a time: subtract off the projection onto everything already built, then normalize what is left.

One step of this process is applied by hand below: keep gram_u1 as is, then remove the part of gram_v2 that points in gram_u1's direction (its projection), leaving a vector orthogonal to gram_u1.

In [10]:
gram_u1 = np.array([1,1])
gram_v2 = np.array([1,0])

# keep the first vector, but remove the part of v2 that points in the u1 direction by calculating the projection of v2 onto u1
proj_of_v2 = (np.dot(gram_v2, gram_u1) / np.dot(gram_u1, gram_u1)) * gram_u1

# subtract that projection from v2
gram_u2 = gram_v2 - proj_of_v2

# check the dot product
result = np.isclose(np.dot(gram_u1,gram_u2), 0)

# normalize u2 and u1 (if isn't already unit length)
q1 = gram_u1 / np.linalg.norm(gram_u1)
q2 = gram_u2 / np.linalg.norm(gram_u2)

print("Is u1 and u2 orthogonal: ", result)


Is u1 and u2 orthogonal:  True


### General Gram-Schmidt Function

**Theorem 1.2.20**, generalized to any number of vectors, For each new vector, we subtract its projection onto every previously-built orthonormal vector, then normalize. Repeating this one vector at a time produces a full orthonormal basis spanning the same space as the original vectors.

In [11]:
# Takes a list of linearly independent vectors and returns a list of orthonormal vectors spanning the same space
def gram_schmidt(vectors):
    qs = []
    for a in vectors:
      b = a.copy().astype(float)
      for q_prev in qs:
        b = b - np.dot(a, q_prev) * q_prev
      q = b / np.linalg.norm(b)
      qs.append(q)
    return qs

v1 = np.array([1, 1, 0])
v2 = np.array([1, 0, 1])
v3 = np.array([0, 1, 1])

qs = gram_schmidt([v1, v2, v3])
Q = np.column_stack(qs)
print("Q =\n", Q)

# verify orthonormal: Q.T @ Q should be identity
print("\nQ.T @ Q =\n", Q.T @ Q)
print("\nIs orthonormal:", np.allclose(Q.T @ Q, np.eye(3)))

Q =
 [[ 0.70710678  0.40824829 -0.57735027]
 [ 0.70710678 -0.40824829  0.57735027]
 [ 0.          0.81649658  0.57735027]]

Q.T @ Q =
 [[ 1.00000000e+00  1.03018891e-16  1.51811967e-16]
 [ 1.03018891e-16  1.00000000e+00 -1.59104334e-16]
 [ 1.51811967e-16 -1.59104334e-16  1.00000000e+00]]

Is orthonormal: True


## 1.2.4 Eigenvectors / eigenvalues

An eigenvector is a vector that, when multiplied by a matrix, only gets stretched, shrunk, or flipped, never rotated off its original direction. The scalar factor by which it's stretched or shrunk is its eigenvalue.

### Verify that A @ x = lambda * x

**Definition 1.2.21.** λ is an eigenvalue of A if there exists a nonzero vector x such that Ax = λx; x is the corresponding eigenvector. Here one eigenvalue/eigenvector is pair is returned by np.linalg.eig and confirmed by the defining equation.

In [12]:
vec1 = np.array([2,0])
vec2 = np.array([0, 3])
C = np.column_stack([vec1, vec2])

eigenvalues, eigenvectors = np.linalg.eig(C)

# Pick the first eigenvalue/eigenvector pair
lam = eigenvalues[0]
y = eigenvectors[:,0]

left = C @ y
right = lam * y

res = np.allclose(left, right)

print("A @ x == lambda * x:", res)

A @ x == lambda * x: True


### Verify eigenvectors from different eigenvalues are orthogonal (dot product = 0)

**Theorem 1.2.25.** If A is symmetric, eigenvectors corresponding to distinct eigenvalues are orthogonal. Here D = [[2,1],[1,2]] is symmetric, so its two eigenvectors (from different eigenvalues) should have a dot product of 0.

In [13]:
# Pick a symmetric matrix (A equals its own transpose (A.T))
x1 = np.array([2,1])
x2 = np.array([1,2])

D = np.column_stack([x1, x2])

eigenvalues, eigenvectors = np.linalg.eig(D)

# pull out two eigenvectors that come from different eigenvalues
e1 = eigenvectors[:,0]
e2 = eigenvectors[:,1]

# Run the dot product check
dot = np.dot(e1, e2)
is_orthogonal = np.isclose(dot, 0)

print("Is e1 and e2 orthogonal: ", is_orthogonal)


Is e1 and e2 orthogonal:  True


### Verify the full diagonalization: check that A = P @ D @ P.T where P has the eigenvectors as columns and D is a diagonal matrix of eigenvalues

**Theorem 1.2.26.** Every symmetric matrix A is orthogonally diagonalizable. A = PDP^T, where P's columns are orthonormal eigenvectors of A and D is diagonal with the corresponding eigenvalues.

In [14]:
# Build E
E = np.diag(eigenvalues)

# reconstruct A using P @ E @ P.T
A_reconstructed = eigenvectors @ E @ eigenvectors.T

# check if the reconstruction matches the original A
is_reconstructed = np.allclose(D, A_reconstructed)

print("Is A reconstructed: ", is_reconstructed)

Is A reconstructed:  True
